# Food Spoilage - Single Model Pipeline

Notebook ini membuat satu model untuk memprediksi `tvc` dan `RSL_minutes`, lalu kelas (`Safe/Warning/Danger`) diturunkan dari TVC. Struktur dibuat mirip notebook utama.

## 1. Konfigurasi dan Inisialisasi

Menyiapkan path, feature, target, dan fungsi mapping class dari TVC.

In [7]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score

RANDOM_SEED = 42

def find_project_root():
    cwd = Path().resolve()
    if (cwd / "data" / "processed" / "dataset_prepared.csv").exists():
        return cwd
    if (cwd.parent / "data" / "processed" / "dataset_prepared.csv").exists():
        return cwd.parent
    return cwd

project_root = find_project_root()
DATA_FILE = project_root / "data" / "processed" / "dataset_prepared.csv"
MODEL_OUT = project_root / "models" / "food_model.pkl"

FEATURES = ["Mq-135", "Mq-136", "Temperature", "Humidity", "minutes"]
TARGETS = ["tvc", "RSL_minutes"]

def tvc_to_class(tvc_value):
    if tvc_value >= 5.0:
        return 3
    if tvc_value > 4.0:
        return 2
    return 1

print("Project root:", project_root)
print("Data file:", DATA_FILE)

Project root: C:\Coding\reksti\Reksti_FreshGuard\Food-Spoilage-Prediction-master\Food-Spoilage-Prediction-master
Data file: C:\Coding\reksti\Reksti_FreshGuard\Food-Spoilage-Prediction-master\Food-Spoilage-Prediction-master\data\processed\dataset_prepared.csv


## 2. Pemuatan Data

Memuat dataset processed dan cek ringkas.

In [8]:
df = pd.read_csv(DATA_FILE)
print("Shape:", df.shape)
print(df.head())
print(df[FEATURES + TARGETS].isnull().sum())

Shape: (18389, 8)
   Mq-135  Mq-136  Temperature  Humidity  minutes    tvc  RSL_minutes  class
0      48     317         29.3      62.4        1  2.002          625      1
1      46     318         29.3      62.3        1  2.002          625      1
2      30     302         29.3      62.3        1  2.002          625      1
3      43     303         29.3      62.5        1  2.002          625      1
4      31     296         29.3      62.9        1  2.002          625      1
Mq-135         0
Mq-136         0
Temperature    0
Humidity       0
minutes        0
tvc            0
RSL_minutes    0
dtype: int64


## 3. Pra-pemrosesan

Menyiapkan fitur, target, dan class dari TVC.

In [9]:
df = df.dropna(subset=FEATURES + TARGETS).copy()
df["class"] = df["tvc"].apply(tvc_to_class).astype(int)
print(df["class"].value_counts().sort_index())

class
1     2782
2     2292
3    13315
Name: count, dtype: int64


## 4. Analisis Ringkas

Distribusi class dan statistik target.

In [10]:
print(df[TARGETS].describe())
print("Class distribution:")
print(df["class"].value_counts(normalize=True).sort_index())

                tvc   RSL_minutes
count  18389.000000  18389.000000
mean       5.197441     60.034695
std        0.932441    120.399843
min        2.002000      0.000000
25%        4.257485      0.000000
50%        5.700000      0.000000
75%        5.902000     40.000000
max        6.314000    625.000000
Class distribution:
class
1    0.151286
2    0.124640
3    0.724074
Name: proportion, dtype: float64


## 5. Training Model Tunggal

Model multi-output memprediksi `tvc` dan `RSL_minutes` sekaligus.

In [11]:
X = df[FEATURES].copy()
y = df[TARGETS].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_SEED
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

base_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    max_depth=5,
    learning_rate=0.1,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED,
    verbosity=1,
 )

model = MultiOutputRegressor(base_model)
model.fit(X_train_scaled, y_train)
print("Model trained")

Model trained


## 6. Evaluasi Model

Hitung MAE dan R2 untuk masing-masing target, serta akurasi class dari TVC prediksi.

In [15]:
preds = model.predict(X_test_scaled)
preds = pd.DataFrame(preds, columns=TARGETS)

for target in TARGETS:
    mae = mean_absolute_error(y_test[target], preds[target])
    r2 = r2_score(y_test[target], preds[target])
    print(f"{target} -> MAE: {mae:.4f} | R2: {r2:.4f}")

pred_class = preds["tvc"].apply(tvc_to_class)
true_class = y_test["tvc"].apply(tvc_to_class)
true_class = y_test["tvc"].apply(tvc_to_class).reset_index(drop=True)

tvc -> MAE: 0.0037 | R2: 0.9998
RSL_minutes -> MAE: 0.2866 | R2: 1.0000


## 7. Fungsi Inferensi

Fungsi ini dipakai untuk input manual dan menghasilkan tvc, rsl, dan class.

In [16]:
def predict_food(mq135, mq136, temperature, humidity, minutes):
    input_data = pd.DataFrame([[mq135, mq136, temperature, humidity, minutes]], columns=FEATURES)
    input_scaled = scaler.transform(input_data)
    tvc_pred, rsl_pred = model.predict(input_scaled)[0]
    class_id = tvc_to_class(tvc_pred)
    return {
        "tvc": float(tvc_pred),
        "rsl_minutes": float(rsl_pred),
        "class": int(class_id)
    }

print(predict_food(360, 260, 24, 55, 120))

{'tvc': 2.137491464614868, 'rsl_minutes': 575.3981323242188, 'class': 1}


In [17]:
# Cell: Export Model dan Scaler
import joblib
from pathlib import Path

# Pastikan folder 'models' ada
MODEL_DIR = project_root / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Definisikan path untuk model dan scaler
MODEL_PATH = MODEL_DIR / "food_model.pkl"
SCALER_PATH = MODEL_DIR / "scaler.pkl"

# Simpan model
joblib.dump(model, MODEL_PATH)
# Simpan scaler (penting agar preprocessing data baru konsisten dengan data training)
joblib.dump(scaler, SCALER_PATH)

print(f"Model berhasil diekspor ke: {MODEL_PATH}")
print(f"Scaler berhasil diekspor ke: {SCALER_PATH}")

Model berhasil diekspor ke: C:\Coding\reksti\Reksti_FreshGuard\Food-Spoilage-Prediction-master\Food-Spoilage-Prediction-master\models\food_model.pkl
Scaler berhasil diekspor ke: C:\Coding\reksti\Reksti_FreshGuard\Food-Spoilage-Prediction-master\Food-Spoilage-Prediction-master\models\scaler.pkl
